# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [16]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [17]:
# Desarrollo de la actividad 1
# Preparar las cuatro columnas de trabajo en eva.
# Mostrar los tipos y los valores que no pudieron convertirse.
# Desarrollo de la actividad 1
# Preparar las cuatro columnas de trabajo en eva
eva["codigo_departamento"] = eva["Código Dane departamento"].str.strip()
eva["anio"] = pd.to_numeric(eva["Año"], errors="coerce").astype("Int64")
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"], errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"], errors="coerce")

total_filas_eva = len(eva)
print(f"Total de filas leídas: {total_filas_eva}")
assert total_filas_eva == 166732, f"Se esperaban 166,732 filas pero hay {total_filas_eva}"

print("\nAusencias generadas por conversión:")
print(f" - anio (nulos): {eva['anio'].isna().sum()}")
print(f" - area_ha (nulos): {eva['area_ha'].isna().sum()}")
print(f" - produccion_t (nulos): {eva['produccion_t'].isna().sum()}")
print("Años:", {str(k): int(v) for k, v in eva["anio"].value_counts(dropna=False).sort_index().items()})
print(eva[["codigo_departamento", "anio", "area_ha", "produccion_t"]].dtypes)


display(eva[["codigo_departamento", "anio", "area_ha", "produccion_t"]].head())


Total de filas leídas: 166732

Ausencias generadas por conversión:
 - anio (nulos): 0
 - area_ha (nulos): 0
 - produccion_t (nulos): 0
Años: {'2019': 20436, '2020': 21511, '2021': 23898, '2022': 24808, '2023': 24909, '2024': 25553, '2025': 25617}
codigo_departamento     object
anio                     Int64
area_ha                float64
produccion_t           float64
dtype: object


,codigo_departamento,anio,area_ha,produccion_t
0,05,2021,8.0,128.00
1,05,2021,8.0,152.00
2,05,2022,8.0,127.44
3,05,2022,8.0,152.00
4,05,2023,7.0,133.00


**Interpretación del resultado:**

Las conversiones numéricas preservan los registros originales intactos. codigo_departamento mantiene los ceros a la izquierda en formato texto, mientras que anio, area_ha y produccion_t admiten análisis cuantitativo registrando adecuadamente sus valores faltantes como nulos numéricos. tambien cabe mencionar que anio es Int64, el cual es entero que admite ausencias, contrario a produccion_t y area_ha que son float64.

## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [18]:
# Desarrollo de la actividad 2
# Construir la máscara y guardar consulta_loc.
# Mostrar los conteos de entrada, selección y diferencia.
# Desarrollo de la actividad 2
# Filtro: Boyacá (15), 2023-2024, área > 0, producción >= 0
mask = (
    (eva["codigo_departamento"] == "15") &
    (eva["anio"].isin([2023, 2024])) &
    (eva["area_ha"] > 0) &
    (eva["produccion_t"] >= 0)
)

consulta_loc = eva.loc[mask].copy()

total_sel = len(consulta_loc)
diferencia = total_filas_eva - total_sel

print(f"Total registros original:   {total_filas_eva}")
print(f"Total registros seleccionados: {total_sel}")
print(f"Diferencia (fuera de alcance): {diferencia}")
print("Selección por año:", {int(k): int(v) for k, v in consulta_loc["anio"].value_counts().sort_index().items()})

Total registros original:   166732
Total registros seleccionados: 4924
Diferencia (fuera de alcance): 161808
Selección por año: {2023: 2436, 2024: 2488}


**Interpretación del resultado:**

La consulta filtra exclusivamente la producción agrícola válida del departamento de Boyacá para el periodo 2023–2024. La diferencia respecto al total original representa datos de otros departamentos, periodos históricos distintos o registros inconsistentes como áreas nulas o negativas. Se constata que el total de los dos años es de 4.924 desglosándose en una selección de 2.436 registros para el año 2023 y 2.488 para el año 2024.

## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [19]:
# Desarrollo de la actividad 3
# Construir consulta_query.
# Comprobar que consulta_loc y consulta_query son equivalentes.
# Desarrollo de la actividad 3
dept_boyaca = "15"
anios_interes = [2023, 2024]

consulta_query = eva.query(
    "codigo_departamento == @dept_boyaca and anio in @anios_interes and area_ha > 0 and produccion_t >= 0",
    engine="python"
).copy()

pd.testing.assert_frame_equal(consulta_loc, consulta_query)
print("consulta_loc:", consulta_loc.shape, "| consulta_query:", consulta_query.shape)
print("✅ Verificación exitosa: consulta_loc y consulta_query son 100% equivalentes.")

consulta_loc: (4924, 22) | consulta_query: (4924, 22)
✅ Verificación exitosa: consulta_loc y consulta_query son 100% equivalentes.


**Interpretación del resultado:**

La prueba de aserción confirma la equivalencia entre el filtrado por máscaras booleanas (.loc) y la sintaxis declarativa (.query). Ambos métodos retornan la misma estructura de datos, permitiendo elegir la sintaxis más legible según la complejidad del proyecto.

## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [20]:
# Desarrollo de la actividad 4
# Crear resumen con registros, total_anio y porcentaje_anual.
# Verificar los totales y los porcentajes por año.
# Desarrollo de la actividad 4
resumen = (
    consulta_loc.groupby(["anio", "Cultivo"], dropna=False)
    .size()
    .reset_index(name="registros")
)

resumen["total_anio"] = resumen.groupby("anio")["registros"].transform("sum")
resumen["porcentaje_anual"] = (resumen["registros"] / resumen["total_anio"]) * 100

resumen = resumen.sort_values(by=["anio", "registros"], ascending=[True, False]).reset_index(drop=True)

display(resumen.head(10))

# Verificación de totales y suma de porcentajes
print("\nComprobación de totales por año:")
for a, grp in resumen.groupby("anio"):
    print(f" - Año {a}: Total registros = {grp['registros'].sum()}, Suma % = {grp['porcentaje_anual'].sum():.2f}%")

,anio,Cultivo,registros,total_anio,porcentaje_anual
0,2023,Papa,239,2436,9.811166
1,2023,Maíz,194,2436,7.963875
2,2023,Arveja,176,2436,7.224959
3,2023,Frijol,162,2436,6.650246
4,2023,Tomate,113,2436,4.638752
5,2023,Cebolla de bulbo,93,2436,3.817734
6,2023,Yuca,82,2436,3.366174
7,2023,Caña,63,2436,2.586207
8,2023,Trigo,58,2436,2.380952
9,2023,Mora,51,2436,2.093596



Comprobación de totales por año:
 - Año 2023: Total registros = 2436, Suma % = 100.00%
 - Año 2024: Total registros = 2488, Suma % = 100.00%


**Interpretación del resultado:**

La agrupación identifica los cultivos con mayor frecuencia de reportes en Boyacá por año. La columna porcentaje_anual confirma que los datos de cada año suman 100%, permitiendo comparar la participación relativa de cada cultivo en la cantidad total de observaciones.

## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [24]:
# Desarrollo de la actividad 5
# Construir y mostrar la tabla dinámica.
# Exportar resumen y explicar dos resultados en la celda siguiente.
# Desarrollo de la actividad 5
# Desarrollo de la actividad 5
import os

# Construcción de la tabla dinámica
tabla_dinamica = pd.pivot_table(
    resumen,
    index="Cultivo",
    columns="anio",
    values="registros",
    fill_value=0
)

display(tabla_dinamica.head(10))

# Exportación del DataFrame resumen a la carpeta OUT


ruta_salida = OUT / "resumen_eva.csv"
resumen.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
assert ruta_salida.is_file()
print(f"\n✅ Archivo exportado exitosamente a: {ruta_salida}")

anio,2023,2024
Cultivo,,
Acelga,5.0,8.0
Achira,4.0,4.0
Agraz - mortiño,3.0,4.0
Aguacate,48.0,52.0
Ahuyama,23.0,26.0
Ajo,7.0,7.0
Ají,1.0,2.0
Apio,0.0,1.0
Arazá,3.0,4.0



✅ Archivo exportado exitosamente a: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02/resumen_eva.csv


**Interpretación del resultado:**

**Conclusión 1: Alta concentración y estabilidad en la estructura de reporte.**
Pocos cultivos concentran la gran mayoría de los reportes agrícolas en Boyacá y la jerarquía entre años se mantiene muy estable. La Papa es el cultivo con mayor número de observaciones en ambos periodos: 239 registros en 2023 (9,81 %) y 237 en 2024 (9,53 %). El grupo de los primeros 5 principales (papa, maíz, frijol, arveja y tomate) acumula el 36,29 % de las observaciones en 2023 y el 35,77 % en 2024, mientras que los demas cultivos se reparten en frecuencias significativamente menores. Esta constancia en el ranking demuestra una estructura de reporte territorial sólida. Cabe precisar que este indicador mide la frecuencia de reportes por municipio y periodo, no la participación en superficie sembrada ni volumen cosechado por lo que puede que un cultivo con pocos registros representre una mayor extensión en hectáreas.

**Conclusión 2: Interpretación del valor cero como ausencia de filtro y no producción nula.**
Los ceros de la tabla dinámica representan la ausencia de observaciones filtradas en la fuente, no una producción agrícola de cero. De los todos los cultivos que aparecen en el periodo de análisis, la gran mayoria se reportan en ambos ambos años,  sin embargo las excepciones son casos como el Apio (registrado únicamente en 2024). El valor `0` asignado a estas combinaciones indica que no se encontraron filas para ese cultivo en el año respectivo con área cosechada estrictamente mayor a cero. Lo cual puede deberse  a registros con área cosechada igual a cero, a no reporte, o a variaciones en la captura de la fuente EVA, por lo que no prueba la inexistencia física del producto en el departamento.

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.